In [ ]:
# 01 · T4 / 공유 모델 하나 / Easy → Medium → Hard
CFG = {
    # 저장 / 초기 Medium 모델
    'run_name': 'moveboxes_unified_curriculum_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',
    'medium_source_run_name': 'moveboxes_medium_zfocus_v22',
    # T4 환경 / state 데이터
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],
    # 같은 모델 하나
    'seed': 42,
    'num_demos': None,
    'batch_size': 32,
    'lr': 3e-05,
    'amp': True,
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,
    # 단계별 통과와 반복 예산
    'block_iters': 500,
    'max_blocks': {'easy': 6, 'medium': 6, 'hard': 6},
    'plateau_blocks': 2,
    'pass_rate': {'easy': 0.8, 'medium': 0.6, 'hard': 0.5},
    'replay_fraction': 0.3,
    'speed_bonus': 0.5,
    # 전체 에피소드 평가 / 시드 분리
    'tuning_episodes': 8,
    'tuning_seed_start': 310000,
    'confirmation_seed_start': 320000,
    'test_episodes': 8,
    'test_seed_start': 330000,
    'benchmark_episodes': 100,
    'eval_seed_start': 340000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},
    # 학습 / 출력
    'warmup_steps': 50,
    'validation_batches': 4,
    'position_noise': 0.0,
    'console_interval_seconds': 30,
    'team': 'my-team',
}


In [ ]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0,str(PROJECT/'hard'/'code'))
sys.path.insert(0,str(PROJECT/'unified'/'code'))
for name in ('build_unified_notebook','unified_policy','unified_data','unified_experiment'):
    if name in sys.modules: importlib.reload(sys.modules[name])
from build_unified_notebook import CONFIG as UNIFIED_DEFAULTS
CFG = dict(UNIFIED_DEFAULTS, **CFG)
from unified_experiment import UnifiedExperiment, source_bundle
experiment = UnifiedExperiment(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증 / 저장된 결과 복원
# 기존 환경변수 GH_TOKEN → Colab 보안 비밀 → 입력창 순서로 인증합니다.
# Fine-grained token: SongYunu/moveBoxes → Contents: Read and write.
# 이 저장소는 공개이므로 여기에 올린 모델·로그·영상도 공개됩니다.
import os
# 개인 사본에서 직접 지정할 경우 아래 한 줄의 주석을 풀어 사용하세요.
# os.environ['GH_TOKEN'] = '본인 토큰'
experiment.connect()
experiment.show_results()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · 데이터 / Medium 초기 가중치 / 공유 정책 복원
experiment.prepare()


In [ ]:
# 06 · 100회 학습 + 2회 평가로 T4 메모리·예상 시간 측정
_ = experiment.calibrate()


In [ ]:
# 07 · EASY 전체 평가 → 전문가 시연 실패 관련 구간 보강 → 반복
experiment.train_level("easy")


In [ ]:
# 08 · EASY 별도 테스트와 영상
experiment.test("easy")


In [ ]:
# 09 · EASY 통과 후 MEDIUM / 이전 데이터 재사용 / 성공 시연 속도 보너스
experiment.train_level("medium")


In [ ]:
# 10 · MEDIUM 별도 테스트와 영상
experiment.test("medium")


In [ ]:
# 11 · 이전 단계 통과 후 HARD 배치 변화 학습
experiment.train_level("hard")


In [ ]:
# 12 · HARD 별도 테스트와 영상
experiment.test("hard")


In [ ]:
# 13 · 공유 모델 상태 / 단계별 채택 기록
_ = experiment.report()


In [ ]:
# 14 · 세 단계 통과 후 최종 각 100회 / 오래 걸리므로 마지막에 실행
experiment.final_evaluation()


In [ ]:
# 15 · 같은 체크포인트 하나를 쓰는 세 난이도 패키지
_ = experiment.package()
